# 34 — Skill Normalization Engine
**Goal:** Normalize skill names to canonical taxonomy (ESCO/O*NET).

## 1. Three-Tier Normalization

In [ ]:
print('''Skill normalization fallback chain:
Tier 1: Exact match -> ESCO/O*NET taxonomy
         "Python" -> "Python (programming language)"
Tier 2: Fuzzy match -> rapidfuzz (threshold 0.85)
         "PyTorch" -> "PyTorch" (typo tolerance)
Tier 3: Embedding similarity -> sentence-transformers (>0.80)
         "ML" -> "Machine Learning"''')

## 2. Building the Normalizer

In [ ]:
from rapidfuzz import fuzz, process

class SkillNormalizer:
    def __init__(self):
        # Canonical skill taxonomy
        self.taxonomy = {
            "python": "Python (programming language)",
            "tensorflow": "TensorFlow",
            "pytorch": "PyTorch",
            "natural language processing": "Natural Language Processing",
            "nlp": "Natural Language Processing",
            "ml": "Machine Learning",
            "machine learning": "Machine Learning",
            "deep learning": "Deep Learning",
            "aws": "Amazon Web Services",
            "gcp": "Google Cloud Platform",
        }
    
    def normalize(self, raw_skill, threshold=80):
        raw = raw_skill.lower().strip()
        
        # Tier 1: exact
        if raw in self.taxonomy:
            return {"normalized": self.taxonomy[raw], "tier": 1, "confidence": 1.0}
        
        # Tier 2: fuzzy
        best = process.extractOne(raw, list(self.taxonomy.keys()), scorer=fuzz.ratio)
        if best and best[1] >= threshold:
            return {"normalized": self.taxonomy[best[0]], "tier": 2, "confidence": best[1]/100}
        
        # Unknown skill
        return {"normalized": raw_skill, "tier": 0, "confidence": 0.5}

n = SkillNormalizer()
for skill in ["Python", "pytorch", "Tensorflo", "ML", "NLP", "CloudWhiz"]:
    result = n.normalize(skill)
    print(f"  '{skill:12s}' -> {result['normalized']:35s} (tier {result['tier']}, conf {result['confidence']:.2f})")

## 3. Embedding-Based Normalization (Tier 3)

In [ ]:
from sentence_transformers import SentenceTransformer, util
import re

class EmbeddingNormalizer(SkillNormalizer):
    def __init__(self):
        super().__init__()
        try:
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
            self.canonical_texts = list(self.taxonomy.values())
            self.canonical_embs = self.model.encode(self.canonical_texts)
            self.has_embeddings = True
        except:
            self.has_embeddings = False
    
    def normalize_embedding(self, raw_skill, threshold=0.65):
        if not self.has_embeddings:
            return self.normalize(raw_skill)
        
        emb = self.model.encode(raw_skill)
        scores = util.cos_sim(emb, self.canonical_embs)[0]
        best_idx = scores.argmax().item()
        best_score = scores[best_idx].item()
        
        if best_score >= threshold:
            return {"normalized": self.canonical_texts[best_idx], "tier": 3, "confidence": best_score}
        return {"normalized": raw_skill, "tier": 0, "confidence": 0.5}

en = EmbeddingNormalizer()
if en.has_embeddings:
    for skill in ["Python", "Tensorflow", "ML", "NLP", "Cloud computing"]:
        r = en.normalize_embedding(skill)
        print(f"  '{skill:18s}' -> {r['normalized']:35s} (tier {r['tier']}, conf {r['confidence']:.2f})")
else:
    print("SentenceTransformer not available. Install with: pip install sentence-transformers")

## Summary: Three-tier normalization catches exact matches, typos, and semantic variants.